# Building and importing Python Libraries

In the previous chapter, we set up an old web server—something you will often encounter in enterprise environments. Over the years, these projects keep growing, but the underlying technologies and libraries often stay the same. Due to time pressure, limited resources, or simply because "it still works", they remain untouched while continuing to run in production.

One of Python's biggest strengths is that we can easily reuse code from other projects. Most libraries are distributed as PyPI packages and can be integrated into existing applications with just a few steps.

In the following sections, we are going to build our own library called ``PyGuard`` and integrate it into Bob's server to protect some of its vulnerable endpoints.

## Implement the middleware

---

## Setup Bob's server

At first, we are setting up Bob's server (once again)

In [5]:
!rm -rf $HOME/tmp/pyguard_demo && mkdir -p $HOME/tmp/pyguard_demo

In [6]:
!cp -r /home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/projXY_bobs_webserver $HOME/tmp/pyguard_demo

In [7]:
!ls -la $HOME/tmp/pyguard_demo/projXY_bobs_webserver

total 1656
drwxr-xr-x 6 fixcfhu fixcfhu    4096 Jul 30 07:45 .
drwxr-xr-x 3 fixcfhu fixcfhu    4096 Jul 30 07:45 ..
drwxr-xr-x 2 fixcfhu fixcfhu    4096 Jul 30 07:45 .ipynb_checkpoints
-rw-r--r-- 1 fixcfhu fixcfhu    1189 Jul 30 07:45 Dockerfile
-rw-r--r-- 1 fixcfhu fixcfhu     912 Jul 30 07:45 README.md
drwxr-xr-x 3 fixcfhu fixcfhu    4096 Jul 30 07:45 etc
-rw-r--r-- 1 fixcfhu fixcfhu 1644552 Jul 30 07:45 image.png
-rw-r--r-- 1 fixcfhu fixcfhu   13233 Jul 30 07:45 legacy_python_projects.ipynb
drwxr-xr-x 2 fixcfhu fixcfhu    4096 Jul 30 07:45 secrets
drwxr-xr-x 4 fixcfhu fixcfhu    4096 Jul 30 07:45 server


In [8]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && export UV_NATIVE_TLS=1 && uv python install 3.9 && uv python pin 3.9 && uv venv --clear

Pinned `.python-version` to `3.9`
Using CPython 3.9.23
Creating virtual environment at: .venv
Activate with: source .venv/bin/activate


In [9]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m ensurepip

Looking in links: /tmp/tmpr7_ucwtu
Processing /tmp/tmpr7_ucwtu/setuptools-58.1.0-py3-none-any.whl
Processing /tmp/tmpr7_ucwtu/pip-23.0.1-py3-none-any.whl


In [10]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip install -r server/requirements.txt

  Using cached requests-2.0.0-py2.py3-none-any.whl (391 kB)
  Using cached urllib3-1.7.1.tar.gz (67 kB)
  Preparing metadata (setup.py) ... done
  Using cached certifi-2015.04.28-py2.py3-none-any.whl (373 kB)
  Using cached chardet-2.1.1.tar.gz (178 kB)
  Preparing metadata (setup.py) ... done
  DEPRECATION: urllib3 is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559
  Running setup.py install for urllib3 ... done
  DEPRECATION: chardet is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at htt

In [11]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip freeze

certifi==2015.4.28
chardet==2.1.1
requests==2.0.0
urllib3==1.7.1


In [ ]:
!tree -L 2 -a $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9

In [30]:
# a small test if the server is up and running
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 server/main.py

/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/server/main.py:10: DeprecationWarning: the imp module is deprecated in favour of importlib; see the module's documentation for alternative uses
  import imp
Using modern compatibility mode.
Legacy API Service
Python: 3.9.23 (main, Sep  2 2025, 14:19:32) 
[Clang 20.1.4 ]
Server: legacy-api
Build: 1837
Compatibility: 7
Listening on http://127.0.0.1:8000

^C
Traceback (most recent call last):
  File "/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/server/main.py", line 319, in <module>
    server.serve_forever()
  File "/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9/socketserver.py", line 232, in serve_forever
    ready = selector.select(poll_interval)
  File "/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9/selectors.py", line 416, in select
    fd_event_list = self._selector.poll(timeout)
KeyboardInterrupt


We are opening a new terminal and see if we can reach the download endpoint of the server
```bash
curl http://127.0.0.1:8000/download?file=api-example.json
curl http://127.0.0.1:8000/download?file=../../secrets/database.conf
```

---

## Import PyGuard

Now it is time to integrate our ``PyGuard`` middleware into Bob's server. As we have already seen, the download endpoint exposes an attack surface for path traversal, making it a good place to integrate our new library.

### Adding it as a editable dependency

When you are developing a Python library locally, you usually want to test changes in your target project as quickly as possible. Packaging, publishing, downloading, and reinstalling the library after every small change would create a very slow development cycle.

To avoid that, Python allows us to install a library as an **editable dependency**. Instead of copying the package into the virtual environment, the package manager creates a link to the library's source code. This means every change you make in the library becomes immediately available in the target project without reinstalling it.

In [12]:
# copying PyGuard into our temp folder
!cp -r /home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/projects/projXY2_pyguard $HOME/tmp/pyguard_demo

In [13]:
# lets get an overview about the current PyGuard folder
!tree -a $HOME/tmp/pyguard_demo/projXY2_pyguard

/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
├── .ipynb_checkpoints
│   └── instructions-checkpoint.ipynb
├── README.md
├── instructions.ipynb
├── pyproject.toml
└── src
    └── pyguard
        ├── __init__.py
        ├── exceptions.py
        ├── middleware.py
        ├── models.py
        ├── rules.py
        └── scanner.py

4 directories, 10 files


In [14]:
# now we are going to install pyguard as a editable dependency
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip install -e $HOME/tmp/pyguard_demo/projXY2_pyguard

Obtaining file:///home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pyguard (pyproject.toml) ... done
  Created wheel for pyguard: filename=pyguard-0.1.0-0.editable-py3-none-any.whl size=1187 sha256=39b04eef8260bb3681f62baa3c843c9bb11905b85ef694ff52e41c981d8f184f
  Stored in directory: /tmp/pip-ephem-wheel-cache-s1cvdzob/wheels/97/27/ee/457ce7c4bf4a1e3cb5e7e8a1defdf7d3cc36405df9e9641333
Successfully built pyguard

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [16]:
# let's check the installed libraries of the virtual environment
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip freeze

certifi==2015.4.28
chardet==2.1.1
# Editable install with no version control (pyguard==0.1.0)
-e /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
requests==2.0.0
urllib3==1.7.1


In [1]:
!tree -L 2 -a $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9

/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9
└── site-packages
    ├── __editable__.pyguard-0.1.0.pth
    ├── __pycache__
    ├── _distutils_hack
    ├── _virtualenv.pth
    ├── _virtualenv.py
    ├── certifi
    ├── certifi-2015.04.28.dist-info
    ├── chardet
    ├── chardet-2.1.1-py3.9.egg-info
    ├── distutils-precedence.pth
    ├── dummyserver
    ├── pip
    ├── pip-23.0.1.dist-info
    ├── pkg_resources
    ├── pyguard-0.1.0.dist-info
    ├── requests
    ├── requests-2.0.0.dist-info
    ├── setuptools
    ├── setuptools-58.1.0.dist-info
    ├── urllib3
    └── urllib3-1.7.1-py3.9.egg-info

19 directories, 4 files


In [31]:
!ls $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9/site-packages/pyguard-0.1.0.dist-info

INSTALLER  METADATA  RECORD  REQUESTED	WHEEL  direct_url.json	top_level.txt


In [6]:
!cat $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9/site-packages/pyguard-0.1.0.dist-info/direct_url.json && \
cat $HOME/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9/site-packages/__editable__.pyguard-0.1.0.pth

{"dir_info": {"editable": true}, "url": "file:///home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard"}/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src


In [26]:
!tree -a $HOME/tmp/pyguard_demo/projXY2_pyguard

/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
├── .ipynb_checkpoints
│   └── instructions-checkpoint.ipynb
├── README.md
├── instructions.ipynb
├── pyproject.toml
└── src
    ├── pyguard
    │   ├── __init__.py
    │   ├── __pycache__
    │   │   ├── __init__.cpython-39.pyc
    │   │   ├── exceptions.cpython-39.pyc
    │   │   ├── middleware.cpython-39.pyc
    │   │   ├── models.cpython-39.pyc
    │   │   ├── rules.cpython-39.pyc
    │   │   └── scanner.cpython-39.pyc
    │   ├── exceptions.py
    │   ├── middleware.py
    │   ├── models.py
    │   ├── rules.py
    │   └── scanner.py
    └── pyguard.egg-info
        ├── PKG-INFO
        ├── SOURCES.txt
        ├── dependency_links.txt
        └── top_level.txt

6 directories, 20 files


#### Conclusion

By using the ``-e`` flag, we install ``PyGuard`` as an editable dependency into our virtual environment. As we have seen, this creates several artifacts inside the site-packages directory:

* a ``__editable__.pyguard-0.1.0.pth`` file, which points to the source code of our local PyGuard project,
* a ``pyguard-0.1.0.dist-info`` directory containing metadata about the installation, including a direct_url.json file that references the local project,
* a ``pyguard.egg-info`` directory inside the PyGuard source tree itself.
  
---

### Playing around with imports
Now that the library is installed, let's take a closer look at how imports actually work and experiment with them a bit to get familiar with the overall concept.

In [2]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip install -e $HOME/tmp/pyguard_demo/projXY2_pyguard

Obtaining file:///home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pyguard (pyproject.toml) ... done
  Created wheel for pyguard: filename=pyguard-0.1.0-0.editable-py3-none-any.whl size=1187 sha256=b743861b41ec7f8f51b6ea1d6dc9f78f838b26b4b903820ef1b15011f2eb97b4
  Stored in directory: /tmp/pip-ephem-wheel-cache-1uohxgny/wheels/97/27/ee/457ce7c4bf4a1e3cb5e7e8a1defdf7d3cc36405df9e9641333
Successfully built pyguard
  Attempting uninstall: pyguard
    Found existing installation: pyguard 0.1.0
    Uninstalling pyguard-0.1.0:
      Successfully uninstalled pyguard-0.1.0

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
!python3 -c "import pyguard"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'pyguard'


In [4]:
# now let's try to import it from the venv interpreter
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import pyguard; print(\"done\")"

done


In [34]:
!python3 -c "import sys; print(sys.path)"

['', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python39.zip', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9/lib-dynload', '/home/fixcfhu/.cache/uv/archive-v0/lKpTXibMan_Mf-8uSVQr8/lib/python3.9/site-packages']


In [35]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import sys; print(sys.path)"

['', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python39.zip', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9', '/home/fixcfhu/.local/share/uv/python/cpython-3.9.23-linux-x86_64-gnu/lib/python3.9/lib-dynload', '/home/fixcfhu/tmp/pyguard_demo/projXY_bobs_webserver/.venv/lib/python3.9/site-packages', '/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src']


In [36]:
# We have seen, that the source code folder ``/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src`` is part 
# of the ``PYTHONPATH`` - what happens if we rename it?
!mv /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard_new

In [37]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import pyguard; print(\"done\")"

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'pyguard'


In [38]:
!mv /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard_new /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard

In [39]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import pyguard; print(\"done\")"

done


In [40]:
!cat /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/pyproject.toml

[project]
name = "pyguard"
version = "0.1.0"
requires-python = ">=3.9"

In [ ]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip uninstall pyguard -y

In [ ]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import sys; print(sys.path)"

#### Conclusion

The ``PyGuard`` import only works when we're inside the virtual environment. To better understand why, we took a closer look at Python's import mechanism using the ``sys`` module. In particular, we examined the ``sys.path`` variable, which contains the list of directories that Python searches whenever an import statement is executed.

By inspecting ``sys.path``, we found that the virtual environment's Python interpreter includes the path to our PyGuard source code due to our *editable* installation. When importing the library, Python matches the import name against the corresponding folder name—in our case, pyguard. We also confirmed that this lookup is case-sensitive by renaming the folder, which immediately caused an import error.

---

### Defining imports from library side

The ``__init__.py`` file is Python's most importing file when it comes to the declaration of packages, because only the existence of this file let Python treat the folder as a package that can be imported by other scripts. 

The ``init__.py`` is the first file that is executed when a custom script uses the ``import`` function of the package and the content of the ``__init__.py`` can influence the importing behaviour. 

In [15]:
# we are adding a print statement inside the __init__.py
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import pyguard"

HELLO FROM PYGUARD


In [16]:
# we are renaming the __init__.py and see what happens
!mv /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard/__init__.py /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard/__init__.py.bak

In [17]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import pyguard" && echo "done"

done


In [18]:
!mv /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard/__init__.py.bak /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard/__init__.py

In [19]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "import pyguard" 

HELLO FROM PYGUARD


---

The ``__init__.py`` bears some useful features that we can use to make the life of customers more easy:
* we can define and set metadata information inside it, like ``__version__``, ``__author__``, ``__license__``
* we can use the ``__all__`` statement to predefine the most important functions to be imported
* we can initialize classes like a logger

In [27]:
!cat /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/src/pyguard/__init__.py

print("HELLO FROM PYGUARD")
from .middleware import PyGuardMiddleware
from .models     import Request, ScanResult
from .exceptions import RequestBlocked

__all__ = ["PyGuardMiddleware",
           "Request",
           "ScanResult",
           "RequestBlocked",
           ]


In [25]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -c "from pyguard import PyGuardMiddleware, RequestBlocked" 

HELLO FROM PYGUARD


We are now familiar with the core prinicples of imports in Python. Now it is time to import ``PyGuard`` as a dependency within Bob's server. From our previous analysis we have seen that the ``download`` endpoint bears a critical attacking surface e.g. with a path traversal. A very common strategie from an attacker. 

In [ ]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip install -e $HOME/tmp/pyguard_demo/projXY2_pyguard

The following snippets are integrated into the ``main.py``

```python
from pyguard import PyGuardMiddleware, Request,
from pyguard import Request as GuardRequest
from pyguard import RequestBlocked

guard = PyGuardMiddleware()

...

guard_request = GuardRequest(method="GET",path=self.path,)
guard.before_request(guard_request)

except RequestBlocked:
    ...
```

curl http://127.0.0.1:8000/download?file=api-example.json
curl http://127.0.0.1:8000/download?file=../../secrets/database.conf

#### Conclusion

We have seen that the ``__init__.py`` is the central entrypoint and that we as libary providers can influence the behaviour of our library by the use of the ``__init__.py``. 

In the end, we were able to solve the security gap with ``PyGuard`` and proved how easy it can be to extend the functionality of legacy projects with the import of libraries. 

### Building the package

In [28]:
!cd /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard && uv build --verbose

DEBUG uv 0.8.18
DEBUG Found workspace root: `/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard`
DEBUG Adding root workspace member: `/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard`
DEBUG No Python version file found in ancestors of working directory: /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
DEBUG Searching for Python >=3.9 in virtual environments, managed installations, or search path
DEBUG Found `cpython-3.12.3-linux-x86_64-gnu` at `/home/fixcfhu/repos/ValentinTwin1206/modern-python-devops-egineering/.venv/bin/python3` (active virtual environment)
DEBUG Using request timeout of 30s
DEBUG Not using uv build backend direct build for source tree `.`, failed to parse pyproject.toml: TOML parse error at line 1, column 1
  |
1 | [project]
  | ^
missing field `build-system`

Building source distribution...
DEBUG Using base executable for virtual environment: /usr/bin/python3.12
DEBUG Resolving build requirements
DEBUG Solving with installed Python version: 3.12.3
DEBUG Solving with target

In [30]:
!tree -L 3 -a /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard

/home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard
├── .ipynb_checkpoints
│   └── instructions-checkpoint.ipynb
├── README.md
├── dist
│   ├── .gitignore
│   ├── pyguard-0.1.0-py3-none-any.whl
│   └── pyguard-0.1.0.tar.gz
├── instructions.ipynb
├── pyproject.toml
└── src
    ├── pyguard
    │   ├── __init__.py
    │   ├── __pycache__
    │   ├── exceptions.py
    │   ├── middleware.py
    │   ├── models.py
    │   ├── rules.py
    │   └── scanner.py
    └── pyguard.egg-info
        ├── PKG-INFO
        ├── SOURCES.txt
        ├── dependency_links.txt
        └── top_level.txt

7 directories, 17 files


In [32]:
# uninstalling the editable dependency and install it from dist folder
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip uninstall pyguard -y && .venv/bin/python3 -m pip freeze 

Found existing installation: pyguard 0.1.0
Uninstalling pyguard-0.1.0:
  Successfully uninstalled pyguard-0.1.0
certifi==2015.4.28
chardet==2.1.1
requests==2.0.0
urllib3==1.7.1


In [33]:
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip install /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/dist/pyguard-0.1.0-py3-none-any.whl 

Processing /home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/dist/pyguard-0.1.0-py3-none-any.whl

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [34]:
# uninstalling the editable dependency and install it from dist folder
!cd $HOME/tmp/pyguard_demo/projXY_bobs_webserver && .venv/bin/python3 -m pip freeze 

certifi==2015.4.28
chardet==2.1.1
pyguard @ file:///home/fixcfhu/tmp/pyguard_demo/projXY2_pyguard/dist/pyguard-0.1.0-py3-none-any.whl
requests==2.0.0
urllib3==1.7.1
